In [14]:
import pandas as pd
import os
import datasets

In [15]:
# Path to .json file containing the dataset
json_file = '/projects/iris/rferreir/GeoBenchmark/data/GeoQuery/t.json'

# Path to the directory containing the HF datasets 
output_path = "/projects/iris/rferreir/hub_datasets"

In [16]:
data = pd.read_json(json_file, orient='records').reset_index(drop=True)

In [17]:
train = data[data['split'] == 'train'].drop(columns=['split'])

In [18]:
test = data[data['split'] == 'test'].drop(columns=['split'])

In [19]:
val = data[data['split'] == 'dev'].drop(columns=['split'])

In [20]:
d1 = {}

In [22]:
columns_to_keep = ['question', 'answer']
for cat in ["place", "regression"]:
    test_split = test[test["type"] == cat][columns_to_keep]
    
    train_split = train[train["type"] == cat][columns_to_keep]
    
    val_split = val[val["type"] == cat][columns_to_keep]

    d1[cat] = datasets.DatasetDict({
        "train": datasets.Dataset.from_pandas(train_split).remove_columns("__index_level_0__"),
        "validation": datasets.Dataset.from_pandas(val_split).remove_columns("__index_level_0__"),
        "test": datasets.Dataset.from_pandas(test_split).remove_columns("__index_level_0__")
    })

    print(d1[cat])
    print(d1[cat]['train'].features)

    d1[cat].save_to_disk(os.path.join(output_path, f"GeoQuery_{cat}"))

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 346
    })
    validation: Dataset({
        features: ['question', 'answer'],
        num_rows: 33
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 184
    })
})
{'question': Value('string'), 'answer': List(Value('string'))}


Saving the dataset (1/1 shards): 100%|██████████| 184/184 [00:00<00:00, 45964.98 examples/s]


DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 182
    })
    validation: Dataset({
        features: ['question', 'answer'],
        num_rows: 17
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 89
    })
})
{'question': Value('string'), 'answer': List(Value('float64'))}


Saving the dataset (1/1 shards): 100%|██████████| 89/89 [00:00<00:00, 17857.49 examples/s]


In [24]:
import hashlib
import json

for cat in ["place", "regression"]:
    d2 = datasets.load_dataset("rfr2003/GeoBenchLLM", f"GeoQuery_{cat}")
    # We check that the content of the dataset is the same
    def content_hash(ds):
        h = hashlib.sha256()
        for ex in ds:
            h.update(json.dumps(ex, sort_keys=True).encode())
        return h.hexdigest()
    
    for split in ['train', 'test', 'validation']:
        assert content_hash(d1[cat][split]) == content_hash(d2[split])

Generating test split: 100%|██████████| 89/89 [00:00<00:00, 22422.70 examples/s]
